# Loading the EDX dataset from the raw .emd file

In [1]:
# imports
import numpy as np
import hyperspy.api as hs
import os

### specify the file path

In [2]:
# set the desired number of frames and the home directory
HomePath = "../data"

# HomePath Structure:
# /path/to/directory
#   |-- EMD    (contains the EMD file)
#   |-- NPZ    (initially empty)

EMDPath = os.path.join(HomePath,'EMD')
file_names = os.listdir(EMDPath)
file_names = [name[:-4] for name in file_names if name.endswith('emd')]
print(file_names)

['EDXdataset']


### wrapper function to load EDX data from EMD files

In [3]:
def load_EDX(file_path, first_frame=0, last_frame = None, sum_frames=True, select_type=None, haadf_last_frame = True): 
    s = hs.load(file_path,
                SI_dtype='uint8',
                first_frame=1,
                last_frame=last_frame,
                sum_frames=sum_frames,
                select_type = None,
                load_SI_image_stack = True)
    # search for the right data
    for i in range(len(s)):
        if '2048, 2048' in repr(s[i]) and 'EDSTEMSpectrum' in repr(s[i]):   
            spectrum_idx = i
        elif 'HAADF' in repr(s[i]): 
            haadf_idx = i

    if haadf_last_frame:
        haadf = s[haadf_idx].data[-1,:,:]
    else:
        haadf = s[haadf_idx].data[:,:,:]
        
    EDX_spectrum = s[spectrum_idx].data   
    xray_energies = s[spectrum_idx].axes_manager.signal_axes[0].axis

    return EDX_spectrum, haadf, xray_energies

### Checks on how frames are loaded

In [5]:
file_path = os.path.join(EMDPath,"%s.emd" % file_names[0])

In [ ]:
# Compare starting from frame 0 and frame 1 - last_frame 5
array1 = load_EDX(file_path, first_frame=0, last_frame=5,sum_frames=True)[0]
array2 = load_EDX(file_path, first_frame=1, last_frame=5,sum_frames=True)[0]

print('Are the arrays equal? ', np.array_equal(array1,array2))
del array1, array2

In [ ]:
# Compare starting from frame 0 and frame 1 - last_frame 5 - sum_frames = False
array1 = load_EDX(file_path, first_frame=0, last_frame=5,sum_frames=False)[0]
array2 = load_EDX(file_path, first_frame=1, last_frame=5,sum_frames=False)[0]

print('Are the arrays equal? ', np.array_equal(array1,array2))
print('Dimensions: ', array1.shape,array2.shape)
del array1, array2

In [ ]:
# Compare starting from frame 0 and frame 1 - last_frame None - sum_frames = True
array1 = load_EDX(file_path, first_frame=0, last_frame=None,sum_frames=True)[0]
array2 = load_EDX(file_path, first_frame=1, last_frame=None,sum_frames=True)[0]

print('Are the arrays equal? ', np.array_equal(array1,array2))
print('Dimensions: ', array1.shape,array2.shape)
del array1, array2

### results from the cell below:
- array1 = load_EDX(file_path, first_frame=None, last_frame=0,sum_frames=False) >> error
- array2 = load_EDX(file_path, first_frame=None, last_frame=1,sum_frames=False) >> error
- array1 = load_EDX(file_path, first_frame=None, last_frame=2,sum_frames=False) >> dim: (1, 2048, 2048, 4096)
- array2 = load_EDX(file_path, first_frame=None, last_frame=3,sum_frames=False) >> dim: (2, 2048, 2048, 4096)
- array1 = load_EDX(file_path, first_frame=None, last_frame=4,sum_frames=False) >> dim: (3, 2048, 2048, 4096)
- array2 = load_EDX(file_path, first_frame=None, last_frame=5,sum_frames=False) >> dim: (4, 2048, 2048, 4096)
- array1 = load_EDX(file_path, first_frame=95, last_frame=99,sum_frames=False)  >> dim: 
- array2 = load_EDX(file_path, first_frame=95, last_frame=100,sum_frames=False) >>

  
- array1 = load_EDX(file_path, first_frame=None, last_frame=101,sum_frames=False) >> ValueError: The `last_frame` cannot be greater than the number of frames, 100 for this file.
- array1 = load_EDX(file_path, first_frame=96, last_frame=99,sum_frames=False)[0] >> memory error, kernel restarts

### Conclusion: 
- It's not conclusive how many frames are actually summed w.r.t the parameter `last_frame`.
  

In [ ]:
# Compare starting from frame None to different last frame values 
array1 = load_EDX(file_path, first_frame=96, last_frame=99,sum_frames=False)[0]
array2 = load_EDX(file_path, first_frame=96, last_frame=100,sum_frames=False)[0]

print('Are the arrays equal? ', np.array_equal(array1,array2))
print('Dimensions: ', array1.shape,array2.shape)
del array1, array2

WARNING | RosettaSciIO | The file contains only one spectrum stream (rsciio.emd._emd_velox:590)


### Load from a fixed number of frames

In [ ]:
# load new file and save to numpy
file_path = os.path.join(EMDPath,"%s.emd" % file_name)
s = hs.load(file_path,
            SI_dtype='uint8',
            first_frame=1,
            last_frame=last_frame,
            sum_frames=True,
            select_type = None,
            load_SI_image_stack = True)

# search for the right data
for i in range(len(s)):
    if '(2048, 2048|4096)' in repr(s[i]):   
        spectrum_idx = i
    elif 'HAADF' in repr(s[i]): 
        haadf_idx = i

haadf = s[haadf_idx].data[-1,:,:]      
spectrum = s[spectrum_idx].data   
xray_energies = s[spectrum_idx].axes_manager.signal_axes[0].axis

In [ ]:
frames = [i for i in range(2,102,10)] 

In [ ]:
#frames = [i for i in range(2,102,10)] 
file_name = file_names[0]
print(file_name)
for last_frame in frames:
    # load new file and save to numpy
    file_path = os.path.join(EMDPath,"%s.emd" % file_name)
    s = hs.load(file_path,
                SI_dtype='uint8',
                first_frame=1,
                last_frame=last_frame,
                sum_frames=True,
                select_type = None,
                load_SI_image_stack = True)
    
    # search for the right data
    for i in range(len(s)):
        if '(2048, 2048|4096)' in repr(s[i]):   
            spectrum_idx = i
        elif 'HAADF' in repr(s[i]): 
            haadf_idx = i
    
    haadf = s[haadf_idx].data[-1,:,:]      
    spectrum = s[spectrum_idx].data   
    xray_energies = s[spectrum_idx].axes_manager.signal_axes[0].axis

    out_path = os.path.join(HomePath,'NPZ','%s_%03dframes.npz' % (file_name,last_frame))
    np.savez_compressed(out_path, haadf=haadf, spectrum=spectrum,xray_energies=xray_energies)
    print("Saved for %02d frames" % last_frame)
    #print(file_path)
    #f,ax = plt.subplots()
    #ax.imshow(haadf,cmap='gray_r')
    #plt.show()
    del haadf, spectrum, s, xray_energies